# Puntos de Calor (Incendios) - IDEAM
**Fuente:** IDEAM - Sistema de Monitoreo de Puntos de Calor  
**URL:** https://puntosdecalor.ideam.gov.co/

In [1]:
import requests, pandas as pd, io, urllib3, csv, shutil
import geopandas as gpd
from datetime import date, timedelta
from pathlib import Path

def leer_excel_seguro(path):
    tmp = path.parent / f'_tmp_{path.name}'
    try:
        shutil.copy2(path, tmp)
        return pd.read_excel(tmp, engine='openpyxl')
    finally:
        if tmp.exists(): tmp.unlink()

print('OK')

OK


## 1. Configuracion

In [2]:
TARGET_DATE = date.today() - timedelta(days=1)
DATE_STR    = TARGET_DATE.strftime('%Y-%m-%d')
REGION      = 'colombia'
EXTENT      = '11.781325296112277_-86.94580078125_-1.8234225930141486_-65.43457031250001'
BASE_URL    = 'https://puntosdecalor.ideam.gov.co/'
DOWNLOAD_URL= f'{BASE_URL}download-result/'
OUTPUT_DIR  = Path(r'C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output4\Indicadores\Incendios')
OUTPUT_FILE = OUTPUT_DIR / 'incendios diarios.xlsx'
WEB_CSV     = Path(r'C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\Scripts Python\webpage_climate\data\incendios_ultimo_dia.csv')
DAVIPOLA    = Path(r'C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\Escenarios Cambio Climatico IDEAM IV comunicacion\Mensuales\davipola_dane.xlsx')
print(f'Fecha: {DATE_STR}')
print(f'Archivo: {OUTPUT_FILE}')

Fecha: 2026-04-20
Archivo: C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output4\Indicadores\Incendios\incendios diarios.xlsx


## 2. Descarga

In [3]:
referer = f'{BASE_URL}?from_date={DATE_STR}&to_date={DATE_STR}&region={REGION}&extent=({EXTENT})'
headers = {'Referer': referer, 'User-Agent': 'Mozilla/5.0', 'Accept': 'text/csv,*/*'}
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
resp = requests.get(DOWNLOAD_URL, headers=headers, timeout=60, verify=False)
resp.raise_for_status()
print(f'Status: {resp.status_code}  Bytes: {len(resp.content):,}')
print(resp.text[:300])

Status: 200  Bytes: 85,085
Fecha (UTC-5);Lat;Lon;Fuente;Temperatura (C);Temperatura Alt* (C);RadiaciÃ³n tÃ©rmica (MW);Confianza;Captura (Dia-Noche);Scan - real pixel size (km);Track - real pixel size (km)
2026-04-20 00:23;5,54334;-68,05336;VIIRS-Suomi-NPP;32,5;17,3;1,2;Nominal;N;0,55;0,68
2026-04-20 00:23;5,54708;-68,06116;


## 3. Parseo del CSV

In [4]:
dialect = csv.Sniffer().sniff(resp.text[:2000], delimiters=',;\t|')
sep_det = dialect.delimiter
print(f'Separador: {repr(sep_det)}')
df_new = pd.read_csv(io.StringIO(resp.text), sep=sep_det, decimal=',', encoding='utf-8')
df_new = df_new.loc[:, ~df_new.columns.str.startswith('Unnamed')]
df_new.insert(0, 'fecha_descarga', pd.to_datetime(DATE_STR))
print(f'Filas: {len(df_new):,}  Columnas: {df_new.columns.tolist()}')
df_new.head()

Separador: ';'
Filas: 1,016  Columnas: ['fecha_descarga', 'Fecha (UTC-5)', 'Lat', 'Lon', 'Fuente', 'Temperatura (C)', 'Temperatura Alt* (C)', 'RadiaciÃ³n tÃ©rmica (MW)', 'Confianza', 'Captura (Dia-Noche)', 'Scan - real pixel size (km)', 'Track - real pixel size (km)']


,fecha_descarga,Fecha (UTC-5),Lat,Lon,Fuente,Temperatura (C),Temperatura Alt* (C),RadiaciÃ³n tÃ©rmica (MW),Confianza,Captura (Dia-Noche),Scan - real pixel size (km),Track - real pixel size (km)
0,2026-04-20,2026-04-20 00:23,5.54334,-68.05336,VIIRS-Suomi-NPP,32.5,17.3,1.2,Nominal,N,0.55,0.68
1,2026-04-20,2026-04-20 00:23,5.54708,-68.06116,VIIRS-Suomi-NPP,30.6,16.8,1.5,Nominal,N,0.55,0.68
2,2026-04-20,2026-04-20 00:23,5.54639,-68.05606,VIIRS-Suomi-NPP,35.9,17.0,2.3,Nominal,N,0.55,0.68
3,2026-04-20,2026-04-20 00:23,5.55041,-68.05759,VIIRS-Suomi-NPP,31.2,17.0,1.3,Nominal,N,0.55,0.68
4,2026-04-20,2026-04-20 00:40,11.13576,-72.57142,VIIRS-NOAA-20,29.3,14.4,1.4,Nominal,N,0.51,0.66


## 4. Cruce espacial con municipios (DIVIPOLA)

In [5]:
# Cargar municipios
davipola = pd.read_excel(DAVIPOLA)
gdf_mun = gpd.GeoDataFrame(
    davipola,
    geometry=gpd.points_from_xy(davipola.LONGITUD, davipola.LATITUD),
    crs='EPSG:4326',
).to_crs(epsg=3116)
print(f'Municipios cargados: {len(gdf_mun):,}')

# Detectar columnas lat/lon en df_new
lat_c = next((c for c in df_new.columns if c.lower() in ['lat','latitud']), None)
lon_c = next((c for c in df_new.columns if c.lower() in ['lon','longitud']), None)
print(f'Columna lat: {lat_c}  lon: {lon_c}')

# Crear GeoDataFrame de puntos de calor
gdf_pts = gpd.GeoDataFrame(
    df_new.copy(),
    geometry=gpd.points_from_xy(
        pd.to_numeric(df_new[lon_c], errors='coerce'),
        pd.to_numeric(df_new[lat_c], errors='coerce')
    ),
    crs='EPSG:4326',
).to_crs(epsg=3116).dropna(subset=['geometry'])

# Cruce espacial: asignar municipio a cada punto de calor
# Usamos buffer de 10 km para capturar puntos cercanos al centroide municipal
gdf_mun_buf = gdf_mun.copy()
gdf_mun_buf['geometry'] = gdf_mun_buf.geometry.buffer(10_000)  # 10 km

joined = gpd.sjoin(
    gdf_pts,
    gdf_mun_buf[['COD_MPIO', 'NOM_MPIO', 'NOM_DPTO', 'geometry']],
    how='left',
    predicate='within'
)

# Si un punto cae en varios buffers (raro), tomar el primero
joined = joined[~joined.index.duplicated(keep='first')]

df_new['COD_MPIO'] = joined['COD_MPIO'].values
df_new['NOM_MPIO'] = joined['NOM_MPIO'].values
df_new['NOM_DPTO'] = joined['NOM_DPTO'].values

sin_mpio = df_new['COD_MPIO'].isna().sum()
print(f'Puntos con municipio asignado: {(~df_new["COD_MPIO"].isna()).sum():,}')
print(f'Puntos sin municipio (fuera de Colombia): {sin_mpio:,}')

# Resumen por municipio
mpio_counts = (
    df_new.dropna(subset=['COD_MPIO'])
    .groupby(['COD_MPIO','NOM_MPIO','NOM_DPTO'])
    .size().reset_index(name='n_puntos')
    .sort_values('n_puntos', ascending=False)
)
print(f'\nMunicipios con puntos de calor: {len(mpio_counts):,}')
mpio_counts.head(15)

Municipios cargados: 1,121
Columna lat: Lat  lon: Lon
Puntos con municipio asignado: 289
Puntos sin municipio (fuera de Colombia): 727

Municipios con puntos de calor: 107


,COD_MPIO,NOM_MPIO,NOM_DPTO,n_puntos
27,13468.0,SANTA CRUZ DE MOMPOX,BOLÍVAR,14
6,5240.0,EBÉJICO,ANTIOQUIA,13
4,5154.0,CAUCASIA,ANTIOQUIA,12
18,13074.0,BARRANCO DE LOBA,BOLÍVAR,10
34,13655.0,SAN JACINTO DEL CAUCA,BOLÍVAR,10
30,13549.0,PINILLOS,BOLÍVAR,9
67,44078.0,BARRANCAS,LA GUAJIRA,8
19,13188.0,CICUCO,BOLÍVAR,6
74,47707.0,SANTA ANA,MAGDALENA,6
43,13838.0,TURBANÁ,BOLÍVAR,6


## 5. Guardar en Excel

In [6]:
if OUTPUT_FILE.exists():
    df_ex = leer_excel_seguro(OUTPUT_FILE)
    if 'fecha_descarga' not in df_ex.columns:
        df_ex.insert(0, 'fecha_descarga', pd.NaT)
    df_ex['fecha_descarga'] = pd.to_datetime(df_ex['fecha_descarga'], errors='coerce')
    fechas_ex = df_ex['fecha_descarga'].dropna().dt.date.unique()
    if TARGET_DATE in fechas_ex:
        print(f'AVISO: {DATE_STR} ya existe.')
        df_new_clean = pd.DataFrame(columns=df_ex.columns)
    else:
        df_new_clean = df_new.copy()
        print(f'{DATE_STR} nueva -> {len(df_new_clean):,} filas.')
else:
    df_ex = pd.DataFrame()
    df_new_clean = df_new.copy()
    print('Archivo nuevo.')

if len(df_new_clean) > 0:
    df_comb = pd.concat([df_ex, df_new_clean], ignore_index=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    tmp_out = OUTPUT_FILE.parent / f'_tmp_{OUTPUT_FILE.name}'
    df_comb.to_excel(tmp_out, index=False, engine='openpyxl')
    shutil.move(str(tmp_out), str(OUTPUT_FILE))
    print(f'Guardado: {OUTPUT_FILE}  ({len(df_comb):,} filas)')

AVISO: 2026-04-20 ya existe.


## 6. Estadisticas

In [7]:
if len(df_new) > 0:
    print('='*50)
    print(f'PUNTOS DE CALOR {DATE_STR}: {len(df_new):,}')
    for kw in ['fuente', 'confianza', 'captura']:
        col = next((c for c in df_new.columns if kw in c.lower()), None)
        if col:
            print(f'\n{col}:')
            print(df_new[col].value_counts().to_string())
    if 'NOM_DPTO' in df_new.columns:
        print('\nPor departamento (top 10):')
        print(df_new['NOM_DPTO'].value_counts().head(10).to_string())

PUNTOS DE CALOR 2026-04-20: 1,016

Fuente:
Fuente
VIIRS-NOAA-20      394
VIIRS-Suomi-NPP    320
VIIRS-NOAA-21      192
MODIS-Aqua         107
MODIS-Terra          3

Confianza:
Confianza
Nominal    827
Baja        63
Alta        16
73 %         7
68 %         6
76 %         4
50 %         4
69 %         4
72 %         4
75 %         4
82 %         3
80 %         3
77 %         3
71 %         3
70 %         3
60 %         3
90 %         3
67 %         3
55 %         2
65 %         2
47 %         2
83 %         2
63 %         2
100 %        2
64 %         2
0 %          2
66 %         2
79 %         2
62 %         2
59 %         2
74 %         2
61 %         2
57 %         2
86 %         1
43 %         1
93 %         1
88 %         1
81 %         1
38 %         1
54 %         1
46 %         1
34 %         1
44 %         1
52 %         1
29 %         1
48 %         1
94 %         1
42 %         1
99 %         1
84 %         1
56 %         1
89 %         1
87 %         1
32 %         1
49 

## 7. Exportar CSV para Dashboard web

In [8]:
def norm_conf(v):
    v = str(v).strip()
    if v in ('Alta','High'): return 'Alta'
    if v in ('Baja','Low'): return 'Baja'
    if v in ('Nominal','Medium'): return 'Nominal'
    try:
        pct = int(v.replace('%','').strip())
        return 'Alta' if pct >= 65 else ('Nominal' if pct >= 30 else 'Baja')
    except: return 'Nominal'

df_all = leer_excel_seguro(OUTPUT_FILE)
df_all.columns = df_all.columns.str.strip()
df_all['fecha_descarga'] = pd.to_datetime(df_all['fecha_descarga'], errors='coerce')
ultimo = df_all['fecha_descarga'].max()
df_u = df_all[df_all['fecha_descarga'] == ultimo].copy()

lat_c = next((c for c in df_u.columns if c.lower() in ['lat','latitud']), None)
lon_c = next((c for c in df_u.columns if c.lower() in ['lon','longitud']), None)
fec_c = next((c for c in df_u.columns if 'fecha' in c.lower() and 'utc' in c.lower()), None)
fue_c = next((c for c in df_u.columns if 'fuente' in c.lower()), None)
tmp_c = next((c for c in df_u.columns if 'temperatura' in c.lower() and 'alt' not in c.lower()), None)
con_c = next((c for c in df_u.columns if 'confianza' in c.lower()), None)
cap_c = next((c for c in df_u.columns if 'captura' in c.lower() or 'dia-noche' in c.lower()), None)
mpio_c = 'NOM_MPIO' if 'NOM_MPIO' in df_u.columns else None
dpto_c = 'NOM_DPTO' if 'NOM_DPTO' in df_u.columns else None
cod_c  = 'COD_MPIO' if 'COD_MPIO' in df_u.columns else None

df_web = pd.DataFrame({
    'fecha_hora': df_u[fec_c] if fec_c else str(ultimo.date()),
    'lat': pd.to_numeric(df_u[lat_c], errors='coerce'),
    'lon': pd.to_numeric(df_u[lon_c], errors='coerce'),
    'fuente': df_u[fue_c] if fue_c else '',
    'temp_c': pd.to_numeric(df_u[tmp_c], errors='coerce') if tmp_c else None,
    'confianza': df_u[con_c].apply(norm_conf) if con_c else 'Nominal',
    'captura': df_u[cap_c] if cap_c else '',
    'municipio': df_u[mpio_c] if mpio_c else '',
    'departamento': df_u[dpto_c] if dpto_c else '',
    'cod_mpio': df_u[cod_c] if cod_c else '',
}).dropna(subset=['lat','lon'])

WEB_CSV.parent.mkdir(parents=True, exist_ok=True)
df_web.to_csv(WEB_CSV, index=False, encoding='utf-8-sig')
print(f'Exportado: {WEB_CSV}')
print(f'Puntos: {len(df_web):,}  Fecha: {ultimo.date()}')
print(df_web['confianza'].value_counts().to_string())
if 'departamento' in df_web.columns:
    print('\nPor departamento (top 10):')
    print(df_web['departamento'].value_counts().head(10).to_string())

Exportado: C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\Scripts Python\webpage_climate\data\incendios_ultimo_dia.csv
Puntos: 1,016  Fecha: 2026-04-20
confianza
Nominal    863
Alta        87
Baja        66

Por departamento (top 10):
departamento
    1016
